# 4.05 Arboles Azarosos — Ensamble a Kaggle de las combinaciones 68, 38 y 80

Toma las combinaciones **68, 38 y 80** (elegidas a partir del ranking de `z422` por ser las mejores en la validacion local) y arma un **ensamble combinado**: en cada punto de `PARAM$grabar` (`1, 2, 4, 8, 16, 32` arboles por combinacion), promedia las probabilidades de las 3 combinaciones entre si y sube **un unico submit** a Kaggle por punto. A diferencia de subir cada combinacion por separado, aca el submit final ya es el promedio de las 3, con peso igual (1/3 cada una).

#### Seteo del ambiente en Google Colab

Esta parte se debe correr con el runtime en Python3
<br>Ir al menu, Runtime -> Change Runtime Type -> Runtime type ->  **Python 3**

Conectar la virtual machine donde esta corriendo Google Colab con el  Google Drive, para poder tener persistencia de archivos

In [1]:
# primero establecer el Runtime de Python 3
from google.colab import drive
drive.mount('/content/.drive')

Mounted at /content/.drive


Para correr la siguiente celda es fundamental en Arranque en Frio haber copiado el archivo kaggle.json al Google Drive, en la carpeta indicada en el instructivo

In [2]:
%%shell

mkdir -p "/content/.drive/My Drive/dmeyf"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/dmeyf"   /content/buckets/b1

mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json  ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json


mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets


# defino funcion descargar()
descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/utn2026-b40a/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}

# hago la descarga efectiva, llamando a descargar()
descargar  "dataset_pequeno.csv"

---

Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

limpio el ambiente de R

In [1]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Sun Aug 23 11:21:35 PM 2026"

In [2]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,671290,35.9,1473300,78.7,1473300,78.7
Vcells,1242666,9.5,8388608,64.0,1978712,15.1


In [3]:
# cargo las librerias que necesito
require("data.table")
require("rpart")

Loading required package: data.table


Attaching package: ‘data.table’


The following object is masked from ‘package:base’:

    %notin%


Loading required package: rpart



Aqui debe cargar SU semilla primigenia. Las combinaciones que se ensamblan (68, 38, 80) ya estan fijadas mas abajo en `PARAM$combos_ensamble`.

In [4]:
PARAM <- list()
PARAM$semilla_primigenia <- 346321

PARAM$rpart$cp <- -1 # fijo, igual que en z422/z423

# voy a generar PARAM$num_trees_max arboles por combinacion, igual que z420/z421
PARAM$num_trees_max <- 32

# puntos del ensemble donde se sube UN submit combinado a Kaggle
PARAM$grabar <- c(1, 2, 4, 8, 16, 32)

# combinaciones que se promedian en el ensamble (elegidas del ranking de z422)
PARAM$combos_ensamble <- c(68, 38, 80, 30)

# donde esta el ranking de z422, de donde se leen los hiperparametros de esas combinaciones
PARAM$archivo_grid_origen <- "/content/buckets/b1/exp/exp422/gridsearch_local.txt"

In [5]:
PARAM$archivo_grid_origen

[1] "/content/buckets/b1/exp/exp422/gridsearch_local.txt"

In [6]:
# chequeo cuantos submits va a consumir esto, contra el limite diario de 100
qty_submits_planificados <- length(PARAM$grabar)

message("combinaciones en el ensamble: ", paste(PARAM$combos_ensamble, collapse = ", "))
message("submits planificados: ", qty_submits_planificados, " / 100 por dia")

if (qty_submits_planificados > 100) {
  warning("Esto planifica mas submits que el limite diario, achica PARAM$grabar")
}

combinaciones en el ensamble: 68, 38, 80, 30

submits planificados: 6 / 100 por dia



In [7]:
# carpeta de trabajo
setwd("/content/buckets/b1/exp")
experimento <- "expEnsamble_68_38_80_30"
dir.create(experimento, showWarnings = FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento ))

In [8]:
# lectura del dataset
dataset <- fread("/content/datasets/dataset_pequeno.csv")

# defino los dataset de entrenamiento y aplicacion, igual que z420: 100% de 202107
#  para entrenar, 202109 (sin clase) es donde se aplica y se sube a Kaggle
dtrain <- dataset[foto_mes == 202107]
dfuture <- dataset[foto_mes == 202109]

# arreglo clase_ternaria por algun distraido ""
dfuture[, clase_ternaria := NA ]

campos_buenos <- copy(setdiff(colnames(dtrain), c("clase_ternaria")))

### Hiperparametros de las combinaciones del ensamble (68, 38, 80)

In [9]:
tb_grid_origen <- fread(PARAM$archivo_grid_origen)

# me quedo con el ultimo punto de arbolito (ensemble completo) de cada combinacion,
#  y filtro solo las 3 combinaciones elegidas para el ensamble
tb_top <- tb_grid_origen[arbolito == max(arbolito)]
combos_ensamble <- tb_top[combo_id %in% PARAM$combos_ensamble]
setorder(combos_ensamble, combo_id)
combos_ensamble

combo_id,feature_fraction,cp,minsplit,minbucket,maxdepth,arbolito,ganancia
<int>,<dbl>,<int>,<int>,<int>,<int>,<int>,<dbl>
30,0.5,-1,1000,50,10,8,494333333
38,0.5,-1,500,50,8,8,493666667
68,0.7,-1,500,100,8,8,495583333
80,0.7,-1,200,150,8,8,490000000


### Submit a Kaggle: ensamble combinado (promedio de 68, 38 y 80)

In [11]:
# matriz de probabilidad acumulada, una columna por cada combinacion del ensamble
prob_acumulada_combos <- matrix(0, nrow = nrow(dfuture), ncol = nrow(combos_ensamble))
colnames(prob_acumulada_combos) <- paste0("combo_", combos_ensamble$combo_id)

# archivo de checkpoint, para no repetir submits si se corta la ejecucion
archivo_grid <- "gridsearch_kaggle_ENSAMBLE_2.txt"

if (file.exists(archivo_grid)) {
  tb_grid <- fread(archivo_grid)
} else {
  tb_grid <- data.table(arbolito = integer(), archivo_kaggle = character())
}

for (arbolito in seq(PARAM$num_trees_max)) {

  # entreno un arbol nuevo por cada combinacion del ensamble, en esta vuelta
  for (i in seq_len(nrow(combos_ensamble))) {

    v_combo_id         <- combos_ensamble[i, combo_id]
    v_feature_fraction <- combos_ensamble[i, feature_fraction]
    v_minsplit         <- combos_ensamble[i, minsplit]
    v_minbucket        <- combos_ensamble[i, minbucket]
    v_maxdepth         <- combos_ensamble[i, maxdepth]

    rpart_control <- list(
      cp        = PARAM$rpart$cp,
      minsplit  = v_minsplit,
      minbucket = v_minbucket,
      maxdepth  = v_maxdepth
    )

    # semilla distinta por combinacion y por arbolito, pero reproducible
    set.seed(PARAM$semilla_primigenia * 100 + v_combo_id * 1000 + arbolito)

    qty_campos_a_utilizar <- as.integer(length(campos_buenos) * v_feature_fraction)
    campos_random <- sample(campos_buenos, qty_campos_a_utilizar)
    campos_random <- paste(campos_random, collapse = " + ")
    formulita <- paste0("clase_ternaria ~ ", campos_random)

    modelo <- rpart(formulita, data = dtrain, xval = 0, control = rpart_control)
    prediccion <- predict(modelo, dfuture, type = "prob")

    prob_acumulada_combos[, i] <- prob_acumulada_combos[, i] + prediccion[, "BAJA+2"]
  }

  if (!(arbolito %in% PARAM$grabar)) next
  if (arbolito %in% tb_grid$arbolito) next

  # cada combinacion aporta su propio promedio (prob_acumulada / arbolito),
  #  y despues se promedian las 3 combinaciones entre si (peso igual, 1/3 cada una)
  prob_promedio_por_combo <- prob_acumulada_combos / arbolito
  prob_ensamble <- rowMeans(prob_promedio_por_combo)

  umbral_corte <- 0.025  # mismo umbral que se uso para las combinaciones individuales

  tb_prediccion <- dfuture[, list(numero_de_cliente)]
  tb_prediccion[, Predicted := as.numeric(prob_ensamble > umbral_corte)]

  archivo_kaggle <- paste0("KA4221_ENSAMBLE_68_38_80_arb", sprintf("%.3d", arbolito), ".csv")
  fwrite( tb_prediccion[, list(numero_de_cliente, Predicted)], file = archivo_kaggle, sep = "," )

  # subida a Kaggle
  comando <- "kaggle competitions submit"
  competencia <- "-c utn-2026-inicial"
  arch <- paste("-f", archivo_kaggle)
  mensaje <- paste0("-m 'ensamble combos=", paste(combos_ensamble$combo_id, collapse = "-"),
    " arbolito=", arbolito, "'")
  linea <- paste(comando, competencia, arch, mensaje)
  salida <- system(linea, intern = TRUE)
  cat(salida)

  tb_grid <- rbindlist(list( tb_grid, data.table(
    arbolito = arbolito,
    archivo_kaggle = archivo_kaggle
  )))

  # grabo el checkpoint despues de cada submit, para poder retomar sin gastar submits de nuevo
  fwrite(tb_grid, file = archivo_grid, sep = "\t")
}

In [ ]:
format(Sys.time(), "%a %b %d %X %Y")